In [138]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [140]:
###Load the ann model, scalary , and onehot pickle file
model = load_model('model.keras')

##load the encoder and scalar
with open('Label_encoder_gender.pkl','rb') as file:
    Label_encoder_gender = pickle.load(file)
    
with open('Onehot_encoder_geo.pkl','rb') as file:
    Onehot_encoder_geo = pickle.load(file)

with open('scalar.pkl','rb') as file:
    scalar = pickle.load(file)

In [180]:
### Example input
input_data = {
    'CreditScore' : 608,
    'Geography' : 'Spain',
    'Gender' : 'Male', 
    'Age' : 41, 
    'Tenure' : 1, 
    'Balance' : 0.0, 
    'NumOfProducts' : 0, 
    'HasCrCard' : 0,
    'IsActiveMember' : 0, 
    'EstimatedSalary' : 0.0
}

In [181]:
## for this example , we need to convert geograph to one hot encoder
geo_encoded = Onehot_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded,columns=Onehot_encoder_geo.get_feature_names_out(['Geography']))

C:\Users\des\anaconda3\Lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


In [182]:
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,0.0,0.0,1.0


In [183]:
## combine one hot encoded columns with input data
input_df = pd.DataFrame([input_data])

In [184]:
# Correct gender encoding
input_df['Gender'] = Label_encoder_gender.transform([input_data['Gender']])

# Correct geography encoding (avoid feature name warning)
geo_df = pd.DataFrame([input_data['Geography']], columns=['Geography'])
geo_encoded = Onehot_encoder_geo.transform(geo_df).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=Onehot_encoder_geo.get_feature_names_out(['Geography']))

In [185]:
input_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1 entries, 0 to 0
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CreditScore      1 non-null      int64  
 1   Geography        1 non-null      object 
 2   Gender           1 non-null      int32  
 3   Age              1 non-null      int64  
 4   Tenure           1 non-null      int64  
 5   Balance          1 non-null      float64
 6   NumOfProducts    1 non-null      int64  
 7   HasCrCard        1 non-null      int64  
 8   IsActiveMember   1 non-null      int64  
 9   EstimatedSalary  1 non-null      float64
dtypes: float64(2), int32(1), int64(6), object(1)
memory usage: 208.0+ bytes


In [186]:
input_df = pd.concat([input_df.drop("Geography",axis=1),geo_encoded_df],axis=1)

In [187]:
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,608,1,41,1,0.0,0,0,0,0.0,0.0,0.0,1.0


In [188]:
##scaling
input_scaled = scalar.transform(input_df)
input_scaled

array([[-0.45296331,  0.91324755,  0.19986603, -1.38944225, -1.21847056,
        -2.6418115 , -1.54035103, -1.02583358, -1.74616572, -0.99850112,
        -0.57946723,  1.73494238]])

In [189]:
predictions = model.predict(input_scaled)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step


In [190]:
predictions

array([[0.54496753]], dtype=float32)

In [191]:
predictions[0][0]

0.54496753

In [192]:
if(predictions[0][0]>0.5):
    print("The customer is likely to churn")
else:
    print("The customer is not likely to churn")

The customer is likely to churn
